# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wajiha-Waqar/FlyRankInternship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-09 — Validation and Research Claim Audit

This notebook audits my Week-5 content-refresh prioritization model.

The purpose is not to maximize the reported score. The purpose is to determine whether the Week-5 result remains credible when evaluated with an honest validation design and after checking for feature leakage.

The audit covers:

1. Two findings from the FlyRank research paper and constructive methodology questions.
2. A before/after comparison between a row-level random split and a client-grouped split.
3. A feature and target leakage audit.
4. Real model disagreement and failure examples.
5. A rewrite of claims using cautious language such as observed, measured, directional, and decision-support.
6. A final self-check against the validation checklist.

## 0. Method and audit principles

My modeling lane is content-refresh prioritization: identifying pages that may deserve review using signals available before a future performance outcome.

The Week-5 experiment used Logistic Regression and Random Forest and evaluated them as ranking systems using Precision@K and Average Precision.

For this audit, I will focus on whether the evaluation design supports the conclusions.

The main questions are:

- Are the features available before the outcome?
- Does the target contain information that is also supplied to the model?
- Does the train/test split prevent pages from the same client appearing in both sets?
- Is the positive-class base rate shown alongside the ranking metrics?
- Are the strongest features plausible, or are they suspiciously close to the target definition?
- Where does the model disagree with the baseline?
- Which claims are supported by the evidence, and which claims need to be narrowed?

In [1]:
# Install the packages needed for the notebook.
!pip -q install duckdb pyarrow pandas numpy scikit-learn matplotlib

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Environment ready.")

Environment ready.


In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. In Google Colab, add your Hugging Face token "
        "under Secrets with the name HF_TOKEN."
    )

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face connection configured.")

Hugging Face connection configured.


In [3]:
march_df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

april_df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

print("March shape:", march_df.shape)
print("April shape:", april_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March shape: (3611061, 31)
April shape: (3901060, 31)


In [4]:
def aggregate_month(df):
    df = df.copy()

    df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
    df["gsc_clicks"] = df["gsc_clicks"].fillna(0)

    result = (
        df.groupby(
            ["client_hash_id", "content_hash_id"],
            as_index=False
        )
        .agg(
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_avg_position=("gsc_avg_position", "mean"),
            ga4_pageviews=("ga4_pageviews", "sum"),
            ga4_sessions=("ga4_sessions", "sum"),
            ga4_users=("ga4_users", "sum"),
            ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
            ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
            sessions_organic=("sessions_organic", "sum"),
            sessions_direct=("sessions_direct", "sum"),
            sessions_referral=("sessions_referral", "sum"),
            sessions_social=("sessions_social", "sum"),
            sessions_paid=("sessions_paid", "sum"),
            sessions_ai=("sessions_ai", "sum"),
            ai_chatgpt=("ai_chatgpt", "sum"),
            ai_perplexity=("ai_perplexity", "sum"),
            ai_gemini=("ai_gemini", "sum"),
            ai_copilot=("ai_copilot", "sum"),
            ai_claude=("ai_claude", "sum"),
            ai_meta=("ai_meta", "sum"),
            ai_other=("ai_other", "sum"),
            scroll_events=("scroll_events", "sum")
        )
    )

    result["ctr"] = np.where(
        result["gsc_impressions"] > 0,
        result["gsc_clicks"] / result["gsc_impressions"],
        np.nan
    )

    return result


march_month = aggregate_month(march_df)
april_month = aggregate_month(april_df)

print("March aggregated:", march_month.shape)
print("April aggregated:", april_month.shape)

March aggregated: (176738, 25)
April aggregated: (194760, 25)


In [5]:
panel = march_month.merge(
    april_month[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_march", "_april")
)

print("Panel shape:", panel.shape)
print("Unique clients:", panel["client_hash_id"].nunique())
print("Unique pages:", panel["content_hash_id"].nunique())

Panel shape: (158549, 28)
Unique clients: 46
Unique pages: 158549


In [6]:
panel["ctr_change_pct"] = (
    (panel["ctr_april"] - panel["ctr_march"])
    / panel["ctr_march"]
)

panel["refresh_outcome"] = (
    (panel["gsc_impressions_march"] >= 100)
    &
    (panel["ctr_march"] > 0)
    &
    (panel["ctr_change_pct"] <= -0.20)
).astype(int)

print(panel["refresh_outcome"].value_counts())
print()
print("Positive rate:", panel["refresh_outcome"].mean())

refresh_outcome
0    120915
1     37634
Name: count, dtype: int64

Positive rate: 0.2373651047941015


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window is the strongest stable freshness band. It also reports a very large 361+ day ratio, but explicitly treats that bucket as unstable because the sample is small.

The paper further reports that, within its dataset, 365+ day content that was refreshed within 30 days showed higher health and impressions than the comparison group.

### Methodology question

My constructive methodology question is:

**How exactly were the growth and decline outcomes used to construct the freshness ratios, and how comparable were the groups being compared before and after refresh?**

In particular, I would want to know:

- how the growing and declining pages were defined;
- whether pages had comparable baseline visibility before the refresh;
- how many pages were in each freshness bucket;
- whether the refreshed and non-refreshed pages differed systematically before the refresh;
- whether the comparison is observational rather than causal.

The question is not whether the finding is useful. The question is whether the validation design supports interpreting the measured difference as an association or as evidence that refreshing caused the improvement.

The paper itself appropriately describes the study as observational and states that correlations do not prove causation.

### Finding 2 — AI Traffic: A Different Signal

The paper reports that pages receiving AI referrals behave differently from pages without AI referrals. In the active-content sample, the high-AI group had substantially more impressions while having a weaker average Google position than the no-AI group.

The paper also notes that AI traffic represented only a small proportion of total portfolio sessions.

### Methodology question

My constructive methodology question is:

**How complete and stable is the definition of an AI-referral page, and how much of the observed difference could be explained by differences in page age, topic, visibility, or other underlying characteristics?**

I would want to know:

- which referral sources were classified as AI sources;
- whether all AI referral sources could be observed;
- whether page age was controlled;
- whether topic or intent was controlled;
- whether the same page could move between AI-traffic buckets over time;
- whether the analysis is a cross-sectional association or a longitudinal comparison.

This matters because the observed difference between AI-referral and non-AI-referral pages does not by itself establish that AI referrals caused the difference in search visibility.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

The Week-5 model should not be evaluated only with a row-level random split because multiple pages can belong to the same client.

The before/after comparison therefore uses:

- BEFORE: ordinary row-level random train/test split.
- AFTER: client-grouped train/test split.

The grouped split is more appropriate for testing performance on clients that were not represented in training.

The gap between the two evaluations is itself useful evidence. A large gap would suggest that the easier random split may have benefited from client-specific patterns that do not transfer to unseen clients.

In [7]:
feature_cols_leaky = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "ctr_march"
]

target_col = "refresh_outcome"

print("Features:")
for feature in feature_cols_leaky:
    print(" -", feature)

print("\nTarget:", target_col)

Features:
 - gsc_impressions_march
 - gsc_clicks_march
 - ctr_march

Target: refresh_outcome


In [8]:
def make_logistic():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ])


def make_random_forest():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=10,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])


def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()


def evaluate_ranking(y_true, scores, name):
    result = {
        "model": name,
        "average_precision": average_precision_score(
            y_true,
            scores
        ),
        "base_rate": np.mean(y_true)
    }

    for k in [20, 50, 100]:
        result[f"precision_at_{k}"] = precision_at_k(
            y_true,
            scores,
            k
        )

    return result

## BEFORE — random row split

In [9]:
X = panel[feature_cols_leaky].copy()
y = panel[target_col].copy()

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Random split")
print("Train rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Train positive rate:", y_train_random.mean())
print("Test positive rate:", y_test_random.mean())

Random split
Train rows: 126839
Test rows: 31710
Train positive rate: 0.2373639022698066
Test positive rate: 0.23736991485335857


In [10]:
logistic_random = make_logistic()
rf_random = make_random_forest()

logistic_random.fit(
    X_train_random,
    y_train_random
)

rf_random.fit(
    X_train_random,
    y_train_random
)

logistic_random_scores = logistic_random.predict_proba(
    X_test_random
)[:, 1]

rf_random_scores = rf_random.predict_proba(
    X_test_random
)[:, 1]

print("Random split models trained.")

Random split models trained.


In [11]:
random_test = panel.loc[
    X_test_random.index
].copy()

random_test["baseline_score"] = (
    (random_test["gsc_impressions_march"] >= 500).astype(int)
    +
    (random_test["ctr_march"] < 0.01).astype(int)
)

random_results = [
    evaluate_ranking(
        y_test_random,
        random_test["baseline_score"],
        "Baseline"
    ),
    evaluate_ranking(
        y_test_random,
        logistic_random_scores,
        "Logistic Regression"
    ),
    evaluate_ranking(
        y_test_random,
        rf_random_scores,
        "Random Forest"
    )
]

random_results_df = pd.DataFrame(random_results)

random_results_df

,model,average_precision,base_rate,precision_at_20,precision_at_50,precision_at_100
0,Baseline,0.381273,0.23737,0.45,0.40,0.41
1,Logistic Regression,0.483430,0.23737,0.50,0.38,0.43
2,Random Forest,0.725358,0.23737,0.85,0.86,0.87


## AFTER — grouped client split

In [12]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        panel,
        panel[target_col],
        groups=panel["client_hash_id"]
    )
)

train_grouped = panel.iloc[train_idx].copy()
test_grouped = panel.iloc[test_idx].copy()

print("Grouped split")
print("Train rows:", len(train_grouped))
print("Test rows:", len(test_grouped))
print("Train clients:", train_grouped["client_hash_id"].nunique())
print("Test clients:", test_grouped["client_hash_id"].nunique())

Grouped split
Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10


In [13]:
train_clients = set(train_grouped["client_hash_id"])
test_clients = set(test_grouped["client_hash_id"])

assert train_clients.isdisjoint(test_clients)

print("No client overlap.")
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

No client overlap.
Train clients: 36
Test clients: 10


In [14]:
X_train_grouped = train_grouped[feature_cols_leaky]
X_test_grouped = test_grouped[feature_cols_leaky]

y_train_grouped = train_grouped[target_col]
y_test_grouped = test_grouped[target_col]

logistic_grouped = make_logistic()
rf_grouped = make_random_forest()

logistic_grouped.fit(
    X_train_grouped,
    y_train_grouped
)

rf_grouped.fit(
    X_train_grouped,
    y_train_grouped
)

logistic_grouped_scores = logistic_grouped.predict_proba(
    X_test_grouped
)[:, 1]

rf_grouped_scores = rf_grouped.predict_proba(
    X_test_grouped
)[:, 1]

test_grouped["baseline_score"] = (
    (test_grouped["gsc_impressions_march"] >= 500).astype(int)
    +
    (test_grouped["ctr_march"] < 0.01).astype(int)
)

grouped_results = [
    evaluate_ranking(
        y_test_grouped,
        test_grouped["baseline_score"],
        "Baseline"
    ),
    evaluate_ranking(
        y_test_grouped,
        logistic_grouped_scores,
        "Logistic Regression"
    ),
    evaluate_ranking(
        y_test_grouped,
        rf_grouped_scores,
        "Random Forest"
    )
]

grouped_results_df = pd.DataFrame(grouped_results)

grouped_results_df

,model,average_precision,base_rate,precision_at_20,precision_at_50,precision_at_100
0,Baseline,0.497366,0.357183,0.30,0.40,0.50
1,Logistic Regression,0.664095,0.357183,0.75,0.62,0.66
2,Random Forest,0.770925,0.357183,0.85,0.80,0.84


## Before/after comparison

In [15]:
before_after = pd.concat(
    [
        random_results_df.assign(split="BEFORE — Random"),
        grouped_results_df.assign(split="AFTER — Grouped by Client")
    ],
    ignore_index=True
)

before_after[
    [
        "split",
        "model",
        "precision_at_20",
        "precision_at_50",
        "precision_at_100",
        "average_precision",
        "base_rate"
    ]
]

,split,model,precision_at_20,precision_at_50,precision_at_100,average_precision,base_rate
0,BEFORE — Random,Baseline,0.45,0.40,0.41,0.381273,0.237370
1,BEFORE — Random,Logistic Regression,0.50,0.38,0.43,0.483430,0.237370
2,BEFORE — Random,Random Forest,0.85,0.86,0.87,0.725358,0.237370
3,AFTER — Grouped by Client,Baseline,0.30,0.40,0.50,0.497366,0.357183
4,AFTER — Grouped by Client,Logistic Regression,0.75,0.62,0.66,0.664095,0.357183
5,AFTER — Grouped by Client,Random Forest,0.85,0.80,0.84,0.770925,0.357183


### Interpretation of the before/after comparison

The random split represents the easier evaluation because pages from the same client can appear in both training and testing.

The grouped split is the more demanding evaluation because complete clients are held out from training.

I interpret any performance difference between the two splits as evidence about transfer across client groups, rather than as proof that one model is inherently better.

The grouped result is the more relevant result for the question:

> Can the model prioritize pages for a client whose pages were not represented in training?

I will use the grouped result as the primary result for the validation audit.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audit the final Week-5 feature set using three main leakage categories:

1. Label-derived leakage.
2. Future or overlapping-window leakage.
3. Decision-derived leakage.

I also check whether population selection depends on information from the outcome window.

### 3.1 Timeline audit

The intended prediction timeline is:

**March data → prediction → April outcome**

The model features are:

- March impressions
- March clicks
- March CTR

The target contains:

- March eligibility conditions
- April CTR decline

Therefore, April performance is not directly supplied to the model.

However, this does not completely eliminate leakage because two March features are themselves used to define the target.

## Explicit label-derived leakage test

In [16]:
leakage_table = pd.DataFrame({
    "feature": [
        "gsc_impressions_march",
        "gsc_clicks_march",
        "ctr_march"
    ],
    "used_in_target_definition": [
        "YES — threshold >= 100",
        "INDIRECTLY — contributes to CTR",
        "YES — threshold > 0"
    ],
    "future_information": [
        "No",
        "No",
        "No"
    ],
    "assessment": [
        "Label-derived",
        "Potentially label-derived through CTR",
        "Label-derived"
    ]
})

leakage_table

,feature,used_in_target_definition,future_information,assessment
0,gsc_impressions_march,YES — threshold >= 100,No,Label-derived
1,gsc_clicks_march,INDIRECTLY — contributes to CTR,No,Potentially label-derived through CTR
2,ctr_march,YES — threshold > 0,No,Label-derived


### Leakage finding

The audit identifies label-derived leakage in the Week-5 feature set.

`gsc_impressions_march` and `ctr_march` are not purely independent predictors of the target because both are used directly in the target definition.

`gsc_clicks_march` contributes to `ctr_march`, so it is also connected to the target construction.

Therefore, the Week-5 model's reported predictive performance should be treated as an exploratory modeling result rather than as clean evidence of independent predictive skill.

This is the main methodological limitation discovered by this audit.

In [17]:
population_audit = {
    "panel_construction": "Inner join March and April on client_hash_id + content_hash_id",
    "requires_April_record": True,
    "outcome_window_used_for_population_selection": True,
    "interpretation": (
        "The evaluated population consists of pages observed in both March and April. "
        "This should be disclosed as a population-selection limitation."
    )
}

population_audit

{'panel_construction': 'Inner join March and April on client_hash_id + content_hash_id',
 'requires_April_record': True,
 'outcome_window_used_for_population_selection': True,
 'interpretation': 'The evaluated population consists of pages observed in both March and April. This should be disclosed as a population-selection limitation.'}

### Population-selection limitation

The modeling panel is created through an inner join between March and April observations.

As a result, the evaluation population requires a matching April record.

This does not mean the experiment is unusable, but it means the result should not automatically be generalized to every page that existed in March.

The evaluated population is better described as:

> pages with matching March and April observations in the available dataset.

This limitation should remain visible in the final interpretation.

In [18]:
decision_derived_features = [
    "baseline_score",
    "optimization_flag",
    "health_score",
    "existing_system_prediction"
]

used_decision_features = [
    feature
    for feature in decision_derived_features
    if feature in feature_cols_leaky
]

print("Decision-derived features used as model inputs:")
print(used_decision_features)

if not used_decision_features:
    print("None of the listed decision-derived features are used as model inputs.")

Decision-derived features used as model inputs:
[]
None of the listed decision-derived features are used as model inputs.


In [21]:
panel["deliberately_leaky_feature"] = panel["refresh_outcome"]

leaky_features = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "ctr_march",
    "refresh_outcome"
]

X_train_leak = train_grouped[leaky_features].copy()
X_test_leak = test_grouped[leaky_features].copy()

leak_model = make_random_forest()

leak_model.fit(
    X_train_leak,
    y_train_grouped
)

leak_scores = leak_model.predict_proba(
    X_test_leak
)[:, 1]

print(
    "Average Precision with deliberate leakage:",
    average_precision_score(
        y_test_grouped,
        leak_scores
    )
)

Average Precision with deliberate leakage: 1.0


In [22]:
panel.drop(
    columns=["deliberately_leaky_feature"],
    inplace=True
)

print("Artificial leakage feature removed.")

Artificial leakage feature removed.


In [23]:
rf_model = rf_grouped.named_steps["model"]

importance = pd.DataFrame({
    "feature": feature_cols_leaky,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance
2,ctr_march,0.402722
1,gsc_clicks_march,0.378393
0,gsc_impressions_march,0.218885


### Feature-importance interpretation

The strongest feature is expected to be `ctr_march`.

This is not evidence that March CTR independently causes or predicts the future outcome.

The target itself contains a condition based on March CTR, so the high importance is partly explained by the target construction.

Therefore, the feature importance result reinforces the leakage finding rather than disproving it.

In [24]:
audit_test = test_grouped.copy()

audit_test["rf_score"] = rf_grouped_scores
audit_test["logistic_score"] = logistic_grouped.predict_proba(
    X_test_grouped
)[:, 1]

audit_test["baseline_score"] = (
    (audit_test["gsc_impressions_march"] >= 500).astype(int)
    +
    (audit_test["ctr_march"] < 0.01).astype(int)
)

audit_test["rf_rank"] = (
    audit_test["rf_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

audit_test["baseline_rank"] = (
    audit_test["baseline_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

audit_test[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_clicks_march",
        "ctr_march",
        "ctr_april",
        "ctr_change_pct",
        "refresh_outcome",
        "baseline_score",
        "rf_score",
        "rf_rank"
    ]
].head()

,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,ctr_march,ctr_april,ctr_change_pct,refresh_outcome,baseline_score,rf_score,rf_rank
19613,client_0e1acc6cd57b0eba,content_0395684dbd55cbbb,103,1,0.009709,0.000000,-1.000000,1,1,0.811793,136.0
19614,client_0e1acc6cd57b0eba,content_21e187279d94b9dd,81,1,0.012346,0.000000,-1.000000,0,0,0.017435,10718.0
19615,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,330,1,0.003030,0.012723,3.198473,0,1,0.699320,2062.0
19616,client_0e1acc6cd57b0eba,content_58e1701ea3fca670,221,0,0.000000,0.003165,inf,0,1,0.000000,14494.0
19617,client_0e1acc6cd57b0eba,content_5ac253ff8287ddf7,160,0,0.000000,0.000000,NaN,0,1,0.000983,11341.0


In [25]:
baseline_misses = audit_test[
    (audit_test["refresh_outcome"] == 1)
    &
    (audit_test["baseline_score"] == 0)
].sort_values(
    "rf_score",
    ascending=False
)

baseline_misses[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "ctr_march",
        "ctr_change_pct",
        "baseline_score",
        "rf_score",
        "rf_rank",
        "refresh_outcome"
    ]
].head(10)

,client_hash_id,content_hash_id,gsc_impressions_march,ctr_march,ctr_change_pct,baseline_score,rf_score,rf_rank,refresh_outcome
152384,client_fef1a8f436438636,content_98fac9164cb33b7e,100,0.010000,-1.000000,0,0.803163,155.0,1
150031,client_fef1a8f436438636,content_550a1ae58c77f8d2,100,0.010000,-0.637681,0,0.803163,154.0,1
144124,client_e5c2aa26a8598242,content_6274d20a30484158,149,0.020134,-0.696538,0,0.781437,252.0,1
32084,client_2094c6eb080311d5,content_30f5fd70664dc4e9,140,0.021429,-1.000000,0,0.780738,290.0,1
35233,client_2094c6eb080311d5,content_fbba4047c2e16d1b,132,0.022727,-0.719745,0,0.778011,370.0,1
32015,client_2094c6eb080311d5,content_2c8604020d695389,142,0.014085,-0.275510,0,0.776288,396.0,1
32814,client_2094c6eb080311d5,content_6264e8d216508596,137,0.021898,-0.512771,0,0.775918,404.0,1
31660,client_2094c6eb080311d5,content_16dc713628278346,141,0.014184,-0.586106,0,0.775657,405.0,1
33146,client_2094c6eb080311d5,content_7705835f87c40fa1,141,0.014184,-1.000000,0,0.775657,406.0,1
33843,client_2094c6eb080311d5,content_a5e41795b4250a1a,143,0.013986,-1.000000,0,0.775507,407.0,1


In [26]:
rf_top_100 = audit_test[
    audit_test["rf_rank"] <= 100
]

rf_false_positives = rf_top_100[
    rf_top_100["refresh_outcome"] == 0
].sort_values(
    "rf_score",
    ascending=False
)

rf_false_positives[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_clicks_march",
        "ctr_march",
        "ctr_april",
        "ctr_change_pct",
        "rf_score",
        "rf_rank",
        "refresh_outcome"
    ]
].head(10)

,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,ctr_march,ctr_april,ctr_change_pct,rf_score,rf_rank,refresh_outcome
53399,client_3f0ce4d44fe94f3d,content_3b836a6b25fd8c18,123,1,0.008130,0.015000,0.845000,0.822825,5.0,0
20064,client_0fa64a184f18a4a0,content_57321e34420ff34a,126,1,0.007937,0.008065,0.016129,0.820858,16.0,0
53898,client_3f0ce4d44fe94f3d,content_5cdfa08e32f8ee79,127,1,0.007874,0.009009,0.144144,0.820858,19.0,0
150920,client_fef1a8f436438636,content_6ec251ff4944ea6d,127,1,0.007874,0.019231,1.442308,0.820858,23.0,0
55055,client_3f0ce4d44fe94f3d,content_aebe2f59163a7889,125,1,0.008000,0.007812,-0.023438,0.820638,28.0,0
55171,client_3f0ce4d44fe94f3d,content_b630678441025a49,125,1,0.008000,0.011450,0.431298,0.820638,29.0,0
142930,client_e5c2aa26a8598242,content_02cd6de142e86513,125,1,0.008000,0.007435,-0.070632,0.820638,30.0,0
32755,client_2094c6eb080311d5,content_5da0e2f14a96a9e2,109,1,0.009174,0.007553,-0.176737,0.820182,41.0,0
143358,client_e5c2aa26a8598242,content_2627dc7a8d58ffe0,108,1,0.009259,0.008435,-0.089035,0.820182,47.0,0
33009,client_2094c6eb080311d5,content_6d7a9c1198248db5,107,1,0.009346,0.009091,-0.027273,0.820182,42.0,0


## 4. Failure examples

I inspected disagreements between the fixed Week-4 baseline and the Random Forest ranking.

These examples are useful because a ranking model is intended to support review prioritization rather than produce a perfect binary decision.

I focus on:

1. Positive pages that the baseline ranked poorly.
2. Pages ranked highly by the Random Forest that did not meet the future outcome definition.

These examples show where the learned ranking differs from the simple threshold rule.

However, because the Week-5 target contains label-derived components, these disagreements should not be interpreted as proof that the Random Forest discovered a causal or independently validated pattern.

They are better described as observed model disagreements within the defined dataset and evaluation procedure.

## 5. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 5. Claim rewrite

### Original-style claim

"The Random Forest provides useful ranking improvement over the baseline and can identify pages that should be refreshed."

### Audited claim

"In the evaluated March-April dataset, the Random Forest produced higher observed ranking metrics than the fixed Week-4 baseline under the tested split. However, the Week-5 target definition includes March CTR and March impressions, which are also model features. This creates label-derived leakage, so the result should be treated as an exploratory decision-support signal rather than independent evidence of predictive skill."

### Stronger claim I will NOT make

I will not claim that the model can reliably identify pages that will lose CTR in production.

The current evidence does not establish that.

### Safe conclusion

"The experiment measured whether simple learned rankings were associated with the defined future outcome in this dataset. The grouped evaluation provides a more conservative estimate of performance across unseen clients, but the target-feature overlap limits the strength of the predictive claim."

In [27]:
clean_feature_cols = [
    "gsc_avg_position_march",
    "ga4_pageviews_march",
    "ga4_sessions_march",
    "ga4_users_march",
    "ga4_engaged_sessions_march",
    "ga4_total_engagement_sec_march",
    "sessions_organic_march",
    "sessions_direct_march",
    "sessions_referral_march",
    "sessions_social_march",
    "sessions_paid_march",
    "sessions_ai_march",
    "ai_chatgpt_march",
    "ai_perplexity_march",
    "ai_gemini_march",
    "ai_copilot_march",
    "ai_claude_march",
    "ai_meta_march",
    "ai_other_march",
    "scroll_events_march"
]

available_clean_features = [
    c for c in clean_feature_cols
    if c in panel.columns
]

missing_clean_features = [
    c for c in clean_feature_cols
    if c not in panel.columns
]

print("Available clean features:")
print(available_clean_features)

print("\nMissing features:")
print(missing_clean_features)

Available clean features:
[]

Missing features:
['gsc_avg_position_march', 'ga4_pageviews_march', 'ga4_sessions_march', 'ga4_users_march', 'ga4_engaged_sessions_march', 'ga4_total_engagement_sec_march', 'sessions_organic_march', 'sessions_direct_march', 'sessions_referral_march', 'sessions_social_march', 'sessions_paid_march', 'sessions_ai_march', 'ai_chatgpt_march', 'ai_perplexity_march', 'ai_gemini_march', 'ai_copilot_march', 'ai_claude_march', 'ai_meta_march', 'ai_other_march', 'scroll_events_march']


In [28]:
if len(available_clean_features) == 0:
    print(
        "No clean features were available under the expected names. "
        "Do not continue with the clean model until the March column names are verified."
    )
else:
    X_clean = panel[available_clean_features].copy()
    y_clean = panel[target_col].copy()

    clean_train_idx, clean_test_idx = next(
        GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=RANDOM_STATE
        ).split(
            X_clean,
            y_clean,
            groups=panel["client_hash_id"]
        )
    )

    X_clean_train = X_clean.iloc[clean_train_idx]
    X_clean_test = X_clean.iloc[clean_test_idx]

    y_clean_train = y_clean.iloc[clean_train_idx]
    y_clean_test = y_clean.iloc[clean_test_idx]

    clean_rf = make_random_forest()

    clean_rf.fit(
        X_clean_train,
        y_clean_train
    )

    clean_scores = clean_rf.predict_proba(
        X_clean_test
    )[:, 1]

    clean_result = evaluate_ranking(
        y_clean_test,
        clean_scores,
        "Random Forest — Clean Feature Set"
    )

    pd.DataFrame([clean_result])

No clean features were available under the expected names. Do not continue with the clean model until the March column names are verified.


## 6. Final limitations

This experiment has several limitations.

1. The target definition uses March impressions and March CTR thresholds, while those same variables were included as model features in the Week-5 model. This creates label-derived leakage.

2. March and April records are joined using an inner join, so the evaluation population consists of content with matching observations in both months.

3. The evaluation covers one March feature window and one April outcome window. It therefore does not establish performance across multiple future periods.

4. The grouped split tests transfer across clients, but it does not replace evaluation on a genuinely future time period.

5. The target is a custom operational definition: a March page with sufficient impressions and measurable CTR followed by at least a 20% CTR decline in April. It should not be interpreted as a universal definition of when a page needs a refresh.

6. Model feature importance is predictive evidence within the tested sample, not causal evidence.

7. The results should therefore be treated as directional and decision-support evidence rather than as proof of production-level predictive reliability.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.